# PART 1-1. 기본 EDA

In [16]:
# 기본 구조 확인하기
import pandas as pd
import numpy as np

df = pd.read_csv('hotel_bookings.csv')

print(df.shape)
print(df.columns.tolist())
df.head()

(119390, 32)
['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [17]:
# 날짜 칼럼 재정의
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-'
    + df['arrival_date_month'] + '-'
    + df['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)

df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03,2015-07-01


In [18]:
# 기본 구조
print("기간:", df['arrival_date'].min(), "~", df['arrival_date'].max())
print("호텔 종류:", df['hotel'].unique())

# 결측치 확인
print("\n결측치 있는 컬럼:")
print(df.isnull().sum()[df.isnull().sum() > 0])

기간: 2015-07-01 00:00:00 ~ 2017-08-31 00:00:00
호텔 종류: ['Resort Hotel' 'City Hotel']

결측치 있는 컬럼:
children         4
country        488
agent        16340
company     112593
dtype: int64


#### 1. 결측치 있는 칼럼: 4개
- children 4개 -> 0으로 두기
- country 488개 -> unknown으로 두기
- agent 16340개 -> 결측치가 유의미한지 확인해보기
- company 112593개 -> 칼럼 사용안하기

#### 2. agent 결측치 16,340개 — 유의미한지 확인

**가설:** agent(여행사 ID)가 결측 = 여행사를 안 거친 직접예약?

In [19]:
# agent 결측 vs 있음 별로 market_segment 분포 비교
df['agent_missing'] = df['agent'].isnull()
print(pd.crosstab(df['agent_missing'], df['market_segment'], normalize='index').round(3))

pd.set_option('display.max_columns', None)   # 컬럼 다 보이게
print(pd.crosstab(df['agent_missing'], df['market_segment'], normalize='index').round(3))

# market_segment의 Direct와 agent 결측이 같은 건지 확인
df['is_direct_agent'] = df['agent'].isnull()
df['is_direct_seg'] = (df['market_segment'] == 'Direct')

print(pd.crosstab(df['is_direct_agent'], df['is_direct_seg']))

market_segment  Aviation  Complementary  Corporate  Direct  Groups  \
agent_missing                                                        
False              0.000          0.001      0.007   0.064   0.152   
True               0.013          0.039      0.281   0.368   0.254   

market_segment  Offline TA/TO  Online TA  Undefined  
agent_missing                                        
False                   0.231      0.545        0.0  
True                    0.024      0.021        0.0  
market_segment  Aviation  Complementary  Corporate  Direct  Groups  \
agent_missing                                                        
False              0.000          0.001      0.007   0.064   0.152   
True               0.013          0.039      0.281   0.368   0.254   

market_segment  Offline TA/TO  Online TA  Undefined  
agent_missing                                        
False                   0.231      0.545        0.0  
True                    0.024      0.021        0.0  
is_dir

**확인 1) 결측 vs market_segment 교차**
- agent 결측 그룹: Direct(37%) + Corporate(28%)가 대부분
- agent 있는 그룹: Online TA(55%) + Offline(23%)
- → 결측은 랜덤 누락이 아니라 "중개 여행사 없는 예약"

**확인 2) market_segment와 중복 여부**
- is_direct(agent 결측)와 market_segment='Direct' 교차 → 부분 중복
- Direct인데 agent 있는 경우(6,599), agent 없는데 Direct 아닌 경우(10,333) 존재
- → 두 변수가 안 맞음. market_segment가 더 세밀

**결론:**
- agent 결측을 "직접예약 신호"로 쓰는 건 부적합 (market_segment와 중복 + 불일치)
- agent는 **ID 자체**로만 활용 (여행사별 취소율 차이 존재: 13%~73%)
- 단, 카디널리티 높음(333개) → Part 2에서 타깃 인코딩 예정
- 결측은 0 또는 'None'으로 채움

#### 3. adr (1박당 평균 요금) 확인

In [20]:
# adr 칼럼 결측값 확인 (음수값 있는지 확인)
print(df['adr'].describe().round(2))
print("\nadr 0 이하:", (df['adr'] <= 0).sum())
print("adr 1000 초과:", (df['adr'] > 1000).sum())

print(df[df['adr'] <= 0]['market_segment'].value_counts())

count    119390.00
mean        101.83
std          50.54
min          -6.38
25%          69.29
50%          94.58
75%         126.00
max        5400.00
Name: adr, dtype: float64

adr 0 이하: 1960
adr 1000 초과: 1
market_segment
Complementary    680
Online TA        367
Offline TA/TO    332
Groups           253
Direct           240
Corporate         82
Aviation           6
Name: count, dtype: int64


**1) 값 성격: 예약 시점 가격**
- 취소된 예약(is_canceled=1)에도 adr 값 존재 (취소군 평균 104.96 vs 정상 99.99)
- → 실제 투숙 결과가 아니라 "예약 시 합의된 가격"
- → 취소군이 오히려 비쌈 (가격↑ → 취소 위험↑ 가능성, Part 4에서 검증)

**2) 이상치 존재**
- min -6.38 (음수), max 5400 (극단값 1건)
- adr ≤ 0: 1,960건 / adr > 1000: 1건

**3) 0원 이하의 정체**
- Complementary(무료): 680건 → 정당한 0원
- 유료 채널(Online TA 등): 1,280건 → 데이터 오류로 추정

**결측값 처리 (Part 4 가격분석 시):**
- adr > 0 & adr < 1000 으로 필터 (약 1,961건 제외)
- 무료(Complementary)는 가격 탄력성 분석에 부적합해 제외가 타당
- 단, 취소예측(Part 2)에선 adr 0이어도 유효한 예약 → 안 지움

# Part 1-2. 취소예측 EDA

In [ ]:
#1. lead_time 확인
# lead_time을 기준으로 구간화하여 취소율 확인 -> 구간마다 최대 60% 차이
df['lead_bucket'] = pd.cut(df['lead_time'],
    bins=[-1, 7, 30, 90, 180, 1000],
    labels=['0-7일', '8-30일', '31-90일', '91-180일', '180일+'])
print(df.groupby('lead_bucket', observed=True)['is_canceled'].agg(['mean', 'size']).round(4))

               mean   size
lead_bucket               
0-7일         0.0963  19746
8-30일        0.2786  18960
31-90일       0.3770  29553
91-180일      0.4471  26439
180일+        0.5701  24692


In [ ]:
#2. market_segment 확인
# market_segment별 취소율 확인 -> Groups가 60% 차지
print(df.groupby('market_segment')['is_canceled'].agg(['mean', 'size']).round(4).sort_values('mean'))

                  mean   size
market_segment               
Complementary   0.1306    743
Direct          0.1534  12606
Corporate       0.1873   5295
Aviation        0.2194    237
Offline TA/TO   0.3432  24219
Online TA       0.3672  56477
Groups          0.6106  19811
Undefined       1.0000      2


In [ ]:
#3. deposit_type 확인 -> non refund가 취소율 높음 (역설적)
print(df.groupby('deposit_type')['is_canceled'].agg(['mean','size']).round(4))

                mean    size
deposit_type                
No Deposit    0.2838  104641
Non Refund    0.9936   14587
Refundable    0.2222     162


In [ ]:
#4. previous_cancellations 확인 -> 이전 취소한 고객이 취소율 높음
print(df.groupby(df['previous_cancellations']>0)['is_canceled'].agg(['mean','size']).round(4))

                          mean    size
previous_cancellations                
False                   0.3391  112906
True                    0.9164    6484


#### * 검증해보기
변수들이 너무 강력하게 의미있음
-> 진짜 의미있는건지 다른 변수가 영향을 주는지 확인해봐야함

1. previous_cancellations 확인해보기

In [49]:
# previous_cancellations 값 분포 (0, 1, 2... 어떻게 퍼져있나)
print("=== 값 분포 ===")
print(df['previous_cancellations'].value_counts().head(10))

# 값별 취소율 (1번 취소 vs 여러번 취소)
print("\n=== 값별 취소율 ===")
print(df.groupby('previous_cancellations')['is_canceled'].agg(['mean','size']).head(10).round(3))

# 재방문객 여부와 관계 (previous가 있으려면 과거 예약이 있어야 함)
print("\n=== 재방문객 여부 ===")
print(df.groupby('is_repeated_guest')['previous_cancellations'].agg(['mean','max']).round(3))

=== 값 분포 ===
previous_cancellations
0     112906
1       6051
2        116
3         65
24        48
11        35
4         31
26        26
25        25
6         22
Name: count, dtype: int64

=== 값별 취소율 ===
                         mean    size
previous_cancellations               
0                       0.339  112906
1                       0.944    6051
2                       0.328     116
3                       0.308      65
4                       0.226      31
5                       0.105      19
6                       0.318      22
11                      0.286      35
13                      0.917      12
14                      1.000      14

=== 재방문객 여부 ===
                    mean  max
is_repeated_guest            
0                  0.075   26
1                  0.470   21


- 1번 취소한 이력이 있는 고객들이 95% 취소할정도로 눈에 띄게 취소율이 큼

In [50]:
# previous_cancellations=1 그룹이 뭐와 겹치나
sub = df[df['previous_cancellations']==1]
print("=== deposit_type ===")
print(sub['deposit_type'].value_counts(normalize=True).round(3))
print("\n=== country 상위 5 ===")
print(sub['country'].value_counts().head(5))
print("\n=== market_segment ===")
print(sub['market_segment'].value_counts(normalize=True).round(3))
print("\n=== 이 그룹의 취소율 ===")
print(sub['is_canceled'].mean().round(3))

=== deposit_type ===
deposit_type
Non Refund    0.584
No Deposit    0.416
Name: proportion, dtype: float64

=== country 상위 5 ===
country
PRT    5791
GBR     114
IRL      42
ESP      35
FRA      18
Name: count, dtype: int64

=== market_segment ===
market_segment
Groups           0.535
Offline TA/TO    0.265
Online TA        0.094
Corporate        0.073
Direct           0.028
Complementary    0.004
Aviation         0.001
Name: proportion, dtype: float64

=== 이 그룹의 취소율 ===
0.944


2. deposit_type 확인해보기

In [51]:
# Non Refund 예약의 실제 상태 분해
print(df[df['deposit_type']=='Non Refund']['reservation_status'].value_counts())

# Non Refund가 특정 조건에 몰려있나
print(df[df['deposit_type']=='Non Refund']['lead_time'].describe().round(1))
print(df[df['deposit_type']=='Non Refund']['country'].value_counts().head())


reservation_status
Canceled     14460
Check-Out       93
No-Show         34
Name: count, dtype: int64
count    14587.0
mean       212.9
std        133.3
min          0.0
25%        103.0
50%        183.0
75%        304.0
max        629.0
Name: lead_time, dtype: float64
country
PRT    14176
GBR      138
ESP      131
DEU       44
POL       38
Name: count, dtype: int64


- non refund는 환불없이 취소한 고객이라는 의미가 정확하다.
- **non refund의 99%가 포르투칼 고객이다.**

3. market_segment 확인해보기 -> 사용가능

In [52]:
# Groups에서 agent 1 빼고도 취소율이 높은가?
groups = df[df['market_segment']=='Groups']
print("Groups 전체 취소율:", groups['is_canceled'].mean().round(3))
print("Groups에서 agent 1 제외:", groups[groups['agent']!=1.0]['is_canceled'].mean().round(3))

Groups 전체 취소율: 0.611
Groups에서 agent 1 제외: 0.54


In [53]:
print("=== 전체 국가 분포 ===")
print(df['country'].value_counts().head(10))
print("\n포르투갈 비율:", (df['country']=='PRT').mean().round(3))

=== 전체 국가 분포 ===
country
PRT    48590
GBR    12129
FRA    10415
ESP     8568
DEU     7287
ITA     3766
IRL     3375
BEL     2342
BRA     2224
NLD     2104
Name: count, dtype: int64

포르투갈 비율: 0.407


In [54]:
sub = df[df['previous_cancellations']==1]
print(sub['agent'].value_counts().head(10))
print("고유 여행사 수:", sub['agent'].nunique())

agent
1.0      2426
19.0      342
9.0       339
6.0       311
29.0      181
37.0      158
240.0     142
3.0       137
96.0       79
5.0        78
Name: count, dtype: int64
고유 여행사 수: 86


In [55]:
a1 = df[df['agent']==1.0]
print("agent 1 취소율:", a1['is_canceled'].mean().round(3))
print("agent 1 총 건수:", len(a1))
print("agent 1 중 포르투갈:", (a1['country']=='PRT').mean().round(3))
print("\nagent 1의 deposit_type:")
print(a1['deposit_type'].value_counts(normalize=True).round(3))

agent 1 취소율: 0.734
agent 1 총 건수: 7191
agent 1 중 포르투갈: 0.839

agent 1의 deposit_type:
deposit_type
Non Refund    0.566
No Deposit    0.434
Name: proportion, dtype: float64


In [56]:
# 포르투갈 vs 비포르투갈 취소율 비교
print("포르투갈 취소율:", df[df['country']=='PRT']['is_canceled'].mean().round(3))
print("비포르투갈 취소율:", df[df['country']!='PRT']['is_canceled'].mean().round(3))

포르투갈 취소율: 0.566
비포르투갈 취소율: 0.236


In [57]:
# agent 1 또는 포르투갈 Groups의 시간 분포
a1['arrival_ym'] = pd.to_datetime(a1['arrival_date']).dt.to_period('M')
print(a1.groupby('arrival_ym').size())

arrival_ym
2015-07     913
2015-08    1122
2015-09    1257
2015-10     624
2015-11     116
2015-12     172
2016-01      29
2016-02      72
2016-03     219
2016-04     271
2016-05     382
2016-06     322
2016-07     272
2016-08     230
2016-09     369
2016-10     290
2016-11     119
2016-12      85
2017-01      72
2017-02      70
2017-03      96
2017-04      68
2017-05       1
2017-06      20
Freq: M, dtype: int64


C:\Users\안서영\AppData\Local\Temp\ipykernel_17120\1289214906.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  a1['arrival_ym'] = pd.to_datetime(a1['arrival_date']).dt.to_period('M')


**취소 강력변수 정리:**
- lead_time: 신뢰 (6배, 시간 무관하게 안정적)
- market_segment: 신뢰 (Groups는 agent1 제외해도 54%)
- deposit_type, previous_cancellations: agent1+포르투갈 편향
  → Part 2에서 포함/제외 비교실험으로 영향 측정

# Part 1-3.수요예측 EDA

In [58]:
# 투숙일별 예약 건수 (수요의 원형)
daily = df.groupby('arrival_date').size().reset_index(name='bookings')
print(daily.head())
print("\n기간:", daily['arrival_date'].min(), "~", daily['arrival_date'].max())
print("일수:", len(daily))
print("일별 예약 수 통계:")
print(daily['bookings'].describe().round(1))

  arrival_date  bookings
0   2015-07-01       122
1   2015-07-02        93
2   2015-07-03        56
3   2015-07-04        88
4   2015-07-05        53

기간: 2015-07-01 00:00:00 ~ 2017-08-31 00:00:00
일수: 793
일별 예약 수 통계:
count    793.0
mean     150.6
std       63.3
min       19.0
25%      106.0
50%      147.0
75%      191.0
max      448.0
Name: bookings, dtype: float64


In [59]:
# 월별 예약 추이 - 2015 하반기가 튀는지
df['arrival_ym'] = pd.to_datetime(df['arrival_date']).dt.to_period('M')
monthly = df.groupby('arrival_ym').size()
print(monthly)

arrival_ym
2015-07    2776
2015-08    3889
2015-09    5114
2015-10    4957
2015-11    2340
2015-12    2920
2016-01    2248
2016-02    3891
2016-03    4824
2016-04    5428
2016-05    5478
2016-06    5292
2016-07    4572
2016-08    5063
2016-09    5394
2016-10    6203
2016-11    4454
2016-12    3860
2017-01    3681
2017-02    4177
2017-03    4970
2017-04    5661
2017-05    6313
2017-06    5647
2017-07    5313
2017-08    4925
Freq: M, dtype: int64


In [60]:
# 호텔별 월별 추이
pivot = df.pivot_table(index='arrival_ym', columns='hotel', 
                        values='is_canceled', aggfunc='size')
print(pivot)

# 요일 패턴
df['dow'] = pd.to_datetime(df['arrival_date']).dt.dayofweek  # 0=월, 6=일
print("\n요일별 예약 (0=월요일):")
print(df.groupby('dow').size())

hotel       City Hotel  Resort Hotel
arrival_ym                          
2015-07           1398          1378
2015-08           2480          1409
2015-09           3529          1585
2015-10           3386          1571
2015-11           1235          1105
2015-12           1654          1266
2016-01           1364           884
2016-02           2371          1520
2016-03           3046          1778
2016-04           3561          1867
2016-05           3676          1802
2016-06           3923          1369
2016-07           3131          1441
2016-08           3378          1685
2016-09           3871          1523
2016-10           4219          1984
2016-11           3122          1332
2016-12           2478          1382
2017-01           2372          1309
2017-02           2594          1583
2017-03           3412          1558
2017-04           3919          1742
2017-05           4556          1757
2017-06           3971          1676
2017-07           3559          1754
2

- 호텔별로 상이하기 때문에 2개를 나눠서 봐야한다.

In [62]:
# 호텔별 요일 패턴
dow_hotel = df.groupby(['hotel', 'dow']).size().unstack()
print(dow_hotel)

# 비율로 보기 (규모 다르니 비율이 비교하기 좋음)
print("\n비율(각 호텔 내 요일 분포):")
print(df.groupby(['hotel','dow']).size().groupby(level=0).apply(lambda x: (x/x.sum()).round(3)).unstack())

dow               0     1      2      3      4      5     6
hotel                                                      
City Hotel    11823  9127  11229  13009  13955  10993  9194
Resort Hotel   6348  4872   4910   6245   5676   7062  4947

비율(각 호텔 내 요일 분포):
dow                            0      1      2      3      4      5      6
hotel        hotel                                                        
City Hotel   City Hotel    0.149  0.115  0.142  0.164  0.176  0.139  0.116
Resort Hotel Resort Hotel  0.158  0.122  0.123  0.156  0.142  0.176  0.123


- 호텔별 요일은 크게 변동없음 :금/토 최고, 화 최저

In [61]:
# 투숙일별: 총예약 vs 실제투숙(취소 제외)
demand = df.groupby('arrival_date').agg(
    total_bookings=('is_canceled', 'size'),
    actual_stays=('is_canceled', lambda x: (x==0).sum())
).reset_index()
demand['cancel_rate'] = 1 - demand['actual_stays'] / demand['total_bookings']
print(demand[['total_bookings','actual_stays','cancel_rate']].describe().round(2))

       total_bookings  actual_stays  cancel_rate
count          793.00        793.00       793.00
mean           150.55         94.79         0.34
std             63.32         34.47         0.13
min             19.00         15.00         0.02
25%            106.00         70.00         0.25
50%            147.00         93.00         0.34
75%            191.00        117.00         0.42
max            448.00        255.00         0.87


In [63]:
# 호텔별 월별 추이 + 순수요(취소 제외)
monthly_hotel = df.groupby(['arrival_ym','hotel']).agg(
    total=('is_canceled','size'),
    actual=('is_canceled', lambda x:(x==0).sum())
).reset_index()

for h in ['City Hotel','Resort Hotel']:
    sub = monthly_hotel[monthly_hotel['hotel']==h]
    print(f"\n=== {h} ===")
    print(sub[['arrival_ym','total','actual']].to_string(index=False))


=== City Hotel ===
arrival_ym  total  actual
   2015-07   1398     459
   2015-08   2480    1248
   2015-09   3529    1986
   2015-10   3386    2065
   2015-11   1235     934
   2015-12   1654     986
   2016-01   1364     926
   2016-02   2371    1441
   2016-03   3046    1938
   2016-04   3561    2022
   2016-05   3676    2240
   2016-06   3923    2203
   2016-07   3131    2088
   2016-08   3378    2131
   2016-09   3871    2304
   2016-10   4219    2272
   2016-11   3122    1762
   2016-12   2478    1406
   2017-01   2372    1328
   2017-02   2594    1623
   2017-03   3412    2134
   2017-04   3919    1993
   2017-05   4556    2339
   2017-06   3971    2163
   2017-07   3559    2235
   2017-08   3125    2002

=== Resort Hotel ===
arrival_ym  total  actual
   2015-07   1378    1058
   2015-08   1409    1043
   2015-09   1585    1034
   2015-10   1571    1160
   2015-11   1105     920
   2015-12   1266     961
   2016-01    884     765
   2016-02   1520    1113
   2016-03   1778    1

In [64]:
daily = df.groupby('arrival_date').size().reset_index(name='bookings')
# 상위 10일
print("=== 예약 많은 날 top 10 ===")
print(daily.nlargest(10, 'bookings').to_string(index=False))
# 하위 10일
print("\n=== 예약 적은 날 bottom 10 ===")
print(daily.nsmallest(10, 'bookings').to_string(index=False))

=== 예약 많은 날 top 10 ===
arrival_date  bookings
  2015-12-05       448
  2016-11-07       366
  2015-10-16       356
  2016-10-13       344
  2015-09-18       340
  2017-06-08       337
  2017-03-02       335
  2016-10-28       334
  2015-09-17       331
  2017-04-29       330

=== 예약 적은 날 bottom 10 ===
arrival_date  bookings
  2015-12-13        19
  2015-11-29        20
  2015-12-07        27
  2015-11-15        28
  2015-12-15        29
  2016-01-31        33
  2015-07-28        34
  2015-07-14        35
  2016-01-24        36
  2016-01-17        37


In [65]:
# 리드타임 구간별로 예약이 얼마나 쌓이나
print("=== 리드타임 분포 (예약이 투숙 며칠 전에 이뤄지나) ===")
print(df['lead_time'].describe().round(1))

# 호텔별 평균 리드타임
print("\n호텔별 평균 리드타임:")
print(df.groupby('hotel')['lead_time'].agg(['mean','median']).round(1))

# 리드타임 구간별 예약 비중 + 그 구간의 취소율
df['lead_bucket'] = pd.cut(df['lead_time'], bins=[-1,7,30,90,180,1000],
    labels=['0-7일','8-30일','31-90일','91-180일','180일+'])
print("\n리드타임 구간별 예약비중 & 취소율:")
print(df.groupby('lead_bucket', observed=True).agg(
    비중=('is_canceled', lambda x: round(len(x)/len(df),3)),
    취소율=('is_canceled','mean')
).round(3))

=== 리드타임 분포 (예약이 투숙 며칠 전에 이뤄지나) ===
count    119390.0
mean        104.0
std         106.9
min           0.0
25%          18.0
50%          69.0
75%         160.0
max         737.0
Name: lead_time, dtype: float64

호텔별 평균 리드타임:
               mean  median
hotel                      
City Hotel    109.7    74.0
Resort Hotel   92.7    57.0

리드타임 구간별 예약비중 & 취소율:
                비중    취소율
lead_bucket              
0-7일         0.165  0.096
8-30일        0.159  0.279
31-90일       0.248  0.377
91-180일      0.221  0.447
180일+        0.207  0.570


In [66]:
# 일별 순수요 시계열 만들기 (City만, 예시)
city = df[(df['hotel']=='City Hotel') & (df['is_canceled']==0)]
city_daily = city.groupby('arrival_date').size()

# 자기상관 (어제, 지난주와 얼마나 연관)
print("=== City 순수요 자기상관 ===")
for lag in [1, 7, 14, 30, 365]:
    corr = city_daily.autocorr(lag=lag)
    print(f"lag {lag}일: {corr:.3f}")

=== City 순수요 자기상관 ===
lag 1일: 0.259
lag 7일: 0.420
lag 14일: 0.381
lag 30일: 0.174
lag 365일: 0.133
